In [1]:
import os
import boto3
import sagemaker
from sagemaker.serializers import IdentitySerializer
from sagemaker.deserializers import JSONDeserializer
import pandas as pd
import numpy as np
import time
from datetime import datetime, timezone
from pyathena import connect
from sqlalchemy import create_engine
from pprint import pprint
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import io
import boto3
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
%store -r
%store

Stored variables and their in-db values:
bucket                                     -> 'sagemaker-us-east-1-298748835671'
database_name                              -> 'cat_landmarking'
ingest_create_athena_db_passed             -> True
ingestion_completed                        -> True
landmarks_feature_group_name               -> 'landmarks-feature-group-17-04-03-42'
landmarks_table                            -> 'cat_annotations'
manifest_table                             -> 'image_manifest'
project_prefix                             -> 'cat-landmarks-project'
s3_athena_results_dir                      -> 's3://sagemaker-us-east-1-298748835671/cat-landmar
s3_processed_cats_prefix                   -> 's3://sagemaker-us-east-1-298748835671/cat-landmar
s3_processed_combined_prefix               -> 's3://sagemaker-us-east-1-298748835671/cat-landmar
s3_raw_cats_prefix                         -> 's3://sagemaker-us-east-1-298748835671/cat-landmar
s3_raw_noncats_prefix                      

In [3]:
# Reassigning store variables for IDE convenience
bucket = bucket
database_name = database_name
project_prefix = project_prefix
landmarks_table = landmarks_table
s3_staging_dir = s3_staging_dir
s3_raw_cats_prefix = s3_raw_cats_prefix
landmarks_feature_group_name = landmarks_feature_group_name
manifest_table = manifest_table

In [4]:
s3 = boto3.client("s3")
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sagemaker_session._region_name

boto_session = boto3.Session(region_name=region)
sagemaker_client = boto_session.client(service_name="sagemaker", 
                                       region_name=region)


# SQL connection specification
engine = create_engine(
    f"awsathena+rest://@athena.{region}.amazonaws.com:443/"
    f"{database_name}"
    f"?s3_staging_dir={s3_staging_dir}"
)

s3_athena_results_dir = f"s3://{bucket}/{project_prefix}/athena/results/"

In [5]:
%store s3_athena_results_dir

Stored 's3_athena_results_dir' (str)


### Building Datasets for Modelling

In [6]:
statement_image_manifest = f"""
	select *
	from
		{database_name}.{manifest_table}
"""

image_manifest_df = pd.read_sql(statement_image_manifest, engine)
image_manifest_df.head()

,remote_path,local_path,file_size,width,height,label,ingest_time
0,s3://sagemaker-us-east-1-298748835671/cat-land...,data/raw/noncats/noncat_images/000000098304.jpg,111609,640,424,0,2026-02-17T03:10:10.291755
1,s3://sagemaker-us-east-1-298748835671/cat-land...,data/raw/noncats/noncat_images/000000425988.jpg,137421,640,498,0,2026-02-17T03:10:10.291764
2,s3://sagemaker-us-east-1-298748835671/cat-land...,data/raw/noncats/noncat_images/000000229381.jpg,120545,456,640,0,2026-02-17T03:10:10.291767
3,s3://sagemaker-us-east-1-298748835671/cat-land...,data/raw/noncats/noncat_images/000000393225.jpg,174759,640,428,0,2026-02-17T03:10:10.291769
4,s3://sagemaker-us-east-1-298748835671/cat-land...,data/raw/noncats/noncat_images/000000032779.jpg,154333,480,640,0,2026-02-17T03:10:10.291771


In [7]:
statement_landmarks = f"""
	select *
	from
		{database_name}.{landmarks_table}
"""

landmarks_df = pd.read_sql(statement_landmarks, engine)
landmarks_df["split"].value_counts()

split
train         23991
validation     1001
test            999
Name: count, dtype: int64

In [8]:
# splitting non-cats data
noncats_df = image_manifest_df[image_manifest_df["label"]==0]
cats_df = image_manifest_df[image_manifest_df["label"]==1]

np.random.seed(42)
noncats_df = noncats_df.copy()
noncats_df["split"] = noncats_df["remote_path"].apply(
    lambda _: np.random.choice(["train", "test", "validation"], p=[.8, .1, .1])
)

# creating data manifest from training, testing, and validation
training_manifest_features = ["remote_path", "label", "split"]
training_manifest = pd.concat([noncats_df[training_manifest_features], 
                              landmarks_df[training_manifest_features]])

training_manifest[["label", "split"]].value_counts().sort_index()

label  split     
0      test           1523
       train         12007
       validation     1470
1      test            999
       train         23991
       validation     1001
Name: count, dtype: int64

In [9]:
# Saving local copy of training manifest
training_manifest_local_path = "data/processed/combined/training_manifest_021626.pqt"
training_manifest.to_parquet(training_manifest_local_path)

# Saving training manifest to S3
training_manifest_remote_path = f"{project_prefix}/data/processed/combined/training_manifests/training_manifest_021626.pqt"
s3.upload_file(training_manifest_local_path, bucket, training_manifest_remote_path)

### Modelling for Classification

In [10]:
# Helper function to read in image
def read_image_from_s3_uri(s3_uri: str) -> Image.Image:
    bucket, key = s3_uri.replace("s3://", "").split("/", 1)
    try:
        obj = s3.get_object(Bucket=bucket, Key=key)
    except Exception as e:
        tqdm.write(f"[ERROR]: {e} - Key: {key}")
    
    if obj:
        return Image.open(io.BytesIO(obj["Body"].read())).convert("RGB")
     

# Dataset builder
class CatClsDatasetCls(Dataset):
    def __init__(self, df: pd.DataFrame, image_size=(224,224), 
                 uri_col: str = None, label_col: str = None):
        
        self.df = df.reset_index(drop=True)
        self.image_size = tuple(image_size)
        self.uri_col = uri_col
        self.label_col = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        s3_uri = row[self.uri_col]
        label = int(row[self.label_col])

        img = read_image_from_s3_uri(s3_uri)
        img = img.resize(self.image_size)

        arr = np.array(img).astype(np.float32) / 255.0  # HWC
        x = torch.from_numpy(arr).permute(2, 0, 1)      # CHW
        y = torch.tensor(label, dtype=torch.long)
        return x, y

In [11]:
IMG_SIZE = 224

# Creating Dataset objects for each split
train_ds_cls = CatClsDatasetCls(
    training_manifest[training_manifest["split"]=="train"], 
    image_size=(IMG_SIZE, IMG_SIZE), 
    uri_col="remote_path", 
    label_col="label")

val_ds_cls   = CatClsDatasetCls(
    training_manifest[training_manifest["split"]=="validation"], 
    image_size=(IMG_SIZE, IMG_SIZE), 
    uri_col="remote_path", 
    label_col="label")

test_ds_cls  = CatClsDatasetCls(
    training_manifest[training_manifest["split"]=="test"], 
    image_size=(IMG_SIZE, IMG_SIZE), 
    uri_col="remote_path", 
    label_col="label")

In [17]:
#  Creating batches for training / validation / testing
BATCH_SIZE = 64

train_loader_cls = DataLoader(train_ds_cls, batch_size=BATCH_SIZE, 
                              shuffle=True, num_workers=16, pin_memory=True)

val_loader_cls   = DataLoader(val_ds_cls, batch_size=BATCH_SIZE, 
                              shuffle=False,num_workers=16, pin_memory=True)

test_loader_cls  = DataLoader(test_ds_cls, batch_size=BATCH_SIZE, 
                              shuffle=False,num_workers=16, pin_memory=True)

### Model Training

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes: int = 2, dropout: float = 0.5):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.gap(x).flatten(1)
        x = self.dropout(x)
        return self.fc(x)


def create_model_and_optimizer(num_classes=2, lr=0.001):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = SmallCNN(num_classes=num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5
    )
    return model, criterion, optimizer, scheduler, device


def run_one_epoch(model, loader, criterion, optimizer, device, train_mode: bool, epoch: int, num_epochs: int):
    model.train() if train_mode else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    phase = "Train" if train_mode else "Val"
    batches = len(loader)

    pbar = tqdm(
        enumerate(loader),
        total=batches,
        desc=f"Epoch {epoch}/{num_epochs} [{phase}]",
        leave=False,
        unit="batch",
    )

    batch_start = time.time()

    with torch.set_grad_enabled(train_mode):
        for batch_idx, (xb, yb) in pbar:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)

            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * xb.size(0)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == yb).sum().item()
            total += xb.size(0)

            # Update progress bar with running stats after each batch
            running_loss = total_loss / total
            running_acc = correct / total
            secs_per_batch = (time.time() - batch_start) / (batch_idx + 1)
            batches_left = batches - (batch_idx + 1)
            eta_secs = int(secs_per_batch * batches_left)

            pbar.set_postfix({
                "loss": f"{running_loss:.4f}",
                "acc":  f"{running_acc:.3f}",
                "batch ETA": f"{eta_secs}s",
            })

    return total_loss / max(total, 1), correct / max(total, 1)


def train_model(model, train_loader, val_loader, criterion, 
                optimizer, scheduler, device, num_epochs: int):
    """Outer training loop with per-epoch summary and overall ETA."""
    best_val_loss = float("inf")
    train_start = time.time()

    epoch_pbar = tqdm(range(1, num_epochs + 1), 
                      desc="Overall progress", unit="epoch")

    for epoch in epoch_pbar:
        epoch_start = time.time()

        train_loss, train_acc = run_one_epoch(
            model, train_loader, criterion, optimizer, device,
            train_mode=True, epoch=epoch, num_epochs=num_epochs
        )
        val_loss, val_acc = run_one_epoch(
            model, val_loader, criterion, optimizer, device,
            train_mode=False, epoch=epoch, num_epochs=num_epochs
        )

        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        epoch_secs = time.time() - epoch_start
        elapsed = time.time() - train_start
        epochs_left = num_epochs - epoch
        eta_secs = int(epoch_secs * epochs_left)
        eta_str = time.strftime("%Hh %Mm %Ss", time.gmtime(eta_secs))

        # Mark best model
        improved = "✓ best" if val_loss < best_val_loss else ""
        if val_loss < best_val_loss:
            best_val_loss = val_loss

        # Clean per-epoch summary line
        print(
            f"Epoch {epoch:>3}/{num_epochs} | "
            f"Train loss: {train_loss:.4f}  acc: {train_acc:.3f} | "
            f"Val loss: {val_loss:.4f}  acc: {val_acc:.3f} | "
            f"LR: {current_lr:.2e} | "
            f"Epoch time: {epoch_secs:.1f}s | "
            f"ETA: {eta_str}  {improved}"
        )

        epoch_pbar.set_postfix({"val_loss": f"{val_loss:.4f}", "ETA": eta_str})

    total_time = time.strftime("%Hh %Mm %Ss", time.gmtime(time.time() - train_start))
    print(f"\nTraining complete in {total_time}. Best val loss: {best_val_loss:.4f}")

# Model Training

In [ ]:
num_epochs = 20
model, criterion, optimizer, scheduler, device = create_model_and_optimizer()
print(f"Device: {device}")

metrics = []

for epoch in range(num_epochs):
    train_loss, train_acc = run_one_epoch(
        model, train_loader_cls, criterion, optimizer, device, 
        train_mode=True, epoch=epoch, num_epochs=num_epochs
    )
    
    val_loss, val_acc = run_one_epoch(
        model, val_loader_cls, criterion, optimizer, device, 
        train_mode=False, epoch=epoch, num_epochs=num_epochs
    )
    
    scheduler.step(val_loss)
    
    metrics.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc
    })
    
    print(f"Epoch {epoch}: Train Loss={train_loss:.4f}, "
          f"Train Acc={train_acc:.4f}, Val Loss={val_loss:.4f}, "
          f"Val Acc={val_acc:.4f}")

Device: cuda


Epoch 0: Train Loss=0.3395, Train Acc=0.8368, Val Loss=0.5090, Val Acc=0.7535


Epoch 1/20 [Train]:  20%|█▉        | 112/563 [00:20<00:41, 10.92batch/s, loss=0.2911, acc=0.871, batch ETA=82s]

# Preserving Training Results Data

In [19]:
# Save CNN training metrics locally
metrics_df = pd.DataFrame(metrics)

#local_path = "/tmp/cnn_training_metrics_v1.csv"
#metrics_df.to_csv(local_path, index=False)

#print("Saved locally:", local_path)
metrics_df.tail()


,epoch,train_loss,train_acc,val_loss,val_acc
0,0,0.327229,0.849519,0.531088,0.757588
1,1,0.267268,0.885299,0.497053,0.763658
2,2,0.250105,0.896939,0.488419,0.766896
3,3,0.237169,0.900356,0.462149,0.779442


In [ ]:
#  Upload CNN metrics to S3 for persistence
key = "cat-landmarks-project/benchmarks/cnn_training_metrics_v1.csv"

s3.upload_file(local_path, bucket, key)

print(f"Uploaded to s3://{bucket}/{key}")

Uploaded to s3://sagemaker-us-east-1-549206572067/cat-landmarks-project/benchmarks/cnn_training_metrics_v1.csv


In [ ]:
# Comments:
# - Use instance-safe temp directory
# - Avoid hardcoding home path
model_dir = "/tmp"
model_path = os.path.join(model_dir, "model_cls.pth")

torch.save(model.state_dict(), model_path)

print("Saved trained model to:", model_path)

In [22]:
eval_metrics = []

# Evaluate on test set once after training
te_loss, te_acc = run_one_epoch(model, test_loader_cls, criterion, optimizer, device, 
                                train_mode=False, epoch=1, num_epochs=1)
print("Test loss:", te_loss, "Test acc:", te_acc)
metrics.append({
    "epoch": "test_final",
    "test_loss": te_loss,
    "test_acc": te_acc,
})

eval_metrics_df = pd.DataFrame(eval_metrics)
eval_metrics_df
#metrics_df.to_csv(local_path, index=False)

# Now upload the updated CSV to S3
#s3.upload_file(local_path, bucket, key)

Test loss: 0.4679327235157579 Test acc: 0.789056304520222


""


# Preserving Model Artifact

In [ ]:
# Save the trained model currently in memory after 5 epochs
model_dir = "/tmp"
model_path = os.path.join(model_dir, "model_cls.pth")

#model_path = "/home/sagemaker-user/model_cls.pth"
torch.save(model.state_dict(), model_path)

print("Saved trained model to:", model_path)

Saved trained model to: /tmp/model_cls.pth


## Endpoint creation/ deployment

In [ ]:
# Move model + create tar.gz for SageMaker
model_dir = "/tmp"
model_path = os.path.join(model_dir, "model_cls.pth")

!rm -rf model model_fixed.tar.gz
!mkdir -p model
!cp {model_path} model/
!cp inference.py model/
!tar -czvf model_fixed.tar.gz model

model/
model/model_cls.pth
model/inference.py


In [ ]:
#  Upload packaged model to S3
endpoint_key = "cat-landmarks-project/endpoint/model_fixed.tar.gz"
s3.upload_file("model_fixed.tar.gz", bucket, endpoint_key)
print(f"Uploaded to s3://{bucket}/{endpoint_key}")

Uploaded to s3://sagemaker-us-east-1-549206572067/cat-landmarks-project/endpoint/model_fixed.tar.gz


In [ ]:
# Update an existing endpoint to use the NEW model artifact without increasing instance count
# Creates: new Model -> new EndpointConfig -> UpdateEndpoint
endpoint_name = "pytorch-inference-2026-02-14-00-36-22-952" 
model_name = f"catcls-fixed-model-{int(time.time())}"
endpoint_config_name = f"catcls-fixed-epc-{int(time.time())}"

model_data_url = "s3://sagemaker-us-east-1-549206572067/cat-landmarks-project/endpoint/model_fixed.tar.gz"
image_uri = sagemaker.image_uris.retrieve(
    framework="pytorch",
    region=region,
    version="1.13",
    py_version="py39",
    instance_type="ml.m5.xlarge",
    image_scope="inference",
)

# 1) Create SageMaker Model that points to the fixed artifact
sagemaker_client.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": image_uri,
        "ModelDataUrl": model_data_url,
        "Environment": {
            # optional while debugging
            "SAGEMAKER_MODEL_SERVER_WORKERS": "1"
        },
    },
)

# 2) Create new EndpointConfig (same instance type/count as existing)
sagemaker_client.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": model_name,
        "InitialInstanceCount": 1,
        "InstanceType": "ml.m5.xlarge",
    }],
)

# 3) Update the existing endpoint in-place (no extra quota)
sagemaker_client.update_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=endpoint_config_name,
)

print("UpdateEndpoint started:", endpoint_name)

UpdateEndpoint started: pytorch-inference-2026-02-14-00-36-22-952


In [ ]:
#  Verifying endpoint 
endpoint_name = "pytorch-inference-2026-02-14-00-36-22-952"

while True:
    desc = sagemaker_client.describe_endpoint(EndpointName=endpoint_name)
    status = desc["EndpointStatus"]
    print("Status:", status)
    if status in ("InService", "Failed"):
        print("Reason:", desc.get("FailureReason", ""))
        break
    time.sleep(20)

Status: Updating
Status: Updating
Status: Updating
Status: InService
Reason: 


## Inference

In [ ]:
# Reading a sample the test image bytes from S3
# Use IdentitySerializer so SageMaker sends raw bytes to input_fn
# Expect JSON back from output_fn
key = "cat-landmarks-project/raw/cats/images/CAT_04/00000900_022.jpg"
obj = s3.get_object(Bucket=bucket, Key=key)
payload = obj["Body"].read()

predictor.serializer = IdentitySerializer(content_type="application/x-image")
predictor.deserializer = JSONDeserializer()

result = predictor.predict(payload)
print(result)

{'prediction': 1}


In [ ]:
# Ensure sure predictor is already created and points to deployed endpoint
predictor.serializer = IdentitySerializer(content_type="application/x-image")
predictor.deserializer = JSONDeserializer()

def split_s3_uri(s3_uri: str):
    # s3://bucket/key...
    parts = s3_uri.replace("s3://", "").split("/", 1)
    return parts[0], parts[1]

results = []
sample_df = prod_df.head(50) 

for i, row in sample_df.iterrows():
    s3_uri = row["s3_uri"]
    bucket, key = split_s3_uri(s3_uri)

    payload = s3.get_object(Bucket=bucket, Key=key)["Body"].read()

    pred = predictor.predict(payload)  
    pred_label = int(pred["prediction"])

    
    results.append({
        "row_id": int(i),
        "s3_uri": s3_uri,
        "y_true": int(row["label"]) if "label" in row and pd.notna(row["label"]) else None,
        "y_pred": pred_label,
    })

pred_df = pd.DataFrame(results)
pred_df.head(20)

,row_id,s3_uri,y_true,y_pred
0,0,s3://sagemaker-us-east-1-549206572067/cat-land...,1,1
1,1,s3://sagemaker-us-east-1-549206572067/cat-land...,1,1
2,2,s3://sagemaker-us-east-1-549206572067/cat-land...,1,1
3,3,s3://sagemaker-us-east-1-549206572067/cat-land...,1,1
4,4,s3://sagemaker-us-east-1-549206572067/cat-land...,1,1
5,5,s3://sagemaker-us-east-1-549206572067/cat-land...,1,1
6,6,s3://sagemaker-us-east-1-549206572067/cat-land...,1,1
7,7,s3://sagemaker-us-east-1-549206572067/cat-land...,1,1
8,8,s3://sagemaker-us-east-1-549206572067/cat-land...,1,1
9,9,s3://sagemaker-us-east-1-549206572067/cat-land...,1,1
